In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/us-birth/annotated/checkpoints/post_cell_18.pickle

trying: ['sns']
me:  0
trying: ['time']
me:  0
trying: ['births_by_date']


me:  27
trying: ['years2check']
me:  9
trying: ['quartiles']
me:  13
trying: ['upper_bound']
me:  13
trying: ['yearly_data']
me:  13
trying: ['lower_bound']
me:  13
trying: ['mu']
me:  13
trying: ['birth_data']


me:  13
trying: ['query_string']
me:  13
trying: ['sig']
me:  13
trying: ['np']
me:  0
trying: ['pd']
me:  0
trying: ['plt']
me:  0
trying: ['factor']
me:  2
trying: ['orig_output']
me:  30
trying: ['mapping']
me:  23
trying: ['year']
me:  9
trying: ['births']


me:  27


Declaring variable sns
Declaring variable time
Declaring variable np
Declaring variable pd
Declaring variable plt
Declaring variable factor
Declaring variable years2check
Declaring variable year
Declaring variable quartiles
Declaring variable upper_bound
Declaring variable yearly_data
Declaring variable lower_bound
Declaring variable mu
Declaring variable birth_data
Declaring variable query_string
Declaring variable sig
Declaring variable mapping
Declaring variable births_by_date
Declaring variable births
Declaring variable orig_output


In [4]:
%%RecordEvent
%%time
### cell 19 ###

# Vectorized GPU datetime index conversion without CPU-bound to_datetime
months = births_by_date.index.get_level_values(0)
days   = births_by_date.index.get_level_values(1)
# Cumulative days before each month in 2020 (leap year)
month_offsets = pd.Series([0, 31, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335], dtype="int32")
# Compute day-of-year (0-based)
day_of_year = month_offsets.take(months - 1) + days - 1
# Days from 1970-01-01 to 2020-01-01 = 18262
epoch_days = 18262
# Nanoseconds per day
ns_per_day = 86400 * 10**9
# Build nanosecond timestamps and convert on GPU
ts_ns = (epoch_days + day_of_year) * ns_per_day
births_by_date.index = pd.to_datetime(ts_ns, unit="ns")
births_by_date.head()

CPU times: user 17.2 ms, sys: 8.53 ms, total: 25.8 ms
Wall time: 25.8 ms


,births
2020-01-01,4009.225
2020-01-02,4247.400
2020-01-03,4500.900
2020-01-04,4571.350
2020-01-05,4603.625


In [5]:
%Checkpoint /home/dias-benchmarks/new_notebooks/us-birth/rewritten/q20_rewrite/checkpoints/post_cell_19_try_1.pickle

migration speed (bps): 630464624.4928493
---------------------------
variables to migrate:
births 48
plt 72
epoch_days 28
ts_ns 48
months 48
births_by_date 48
month_offsets 48
days 48
birth_data 48
query_string 98
day_of_year 48
sig 32
ns_per_day 32
quartiles 136
upper_bound 32
yearly_data 48
lower_bound 32
pd 72
mu 32
sns 72
time 72
years2check 120
factor 28
np 72
mapping 360
year 28
orig_output 56
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/new_notebooks/us-birth/rewritten/q20_rewrite/checkpoints/post_cell_19_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['birth_data']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['birth_data']
Intermediate variables ['factor']
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['birth_data']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables ['birth_data']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['birth_data']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables ['birth_data']
Intermediate variables []
Future variables []
Modified dataframes
  birth_data
    Input columns: set()
    Changed columns: set()
    Created columns: {'decade'}
    Deleted columns: set()
======= Cell 6 =======
Input

In [7]:

with open("/home/dias-benchmarks/new_notebooks/us-birth/opt_cell_exec_info_19_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[19], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/us-birth/annotated/checkpoints/post_cell_19.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
